# 07 — Experimento 2: aciertos de cada modelo por tipo de ventana

Para los tres modelos (Qwen2.5-7B, Qwen2.5-14B y Qwen3.5-9B), cuenta los aciertos en cada tipo de ventana **según tus etiquetas**: anomalía, anomalía mitigada y funcionamiento normal. Lo difícil en estos datos es no marcarlo todo como anomalía; en particular, reconocer una **anomalía mitigada** (hay errores, pero los mecanismos de recuperación los corrigen y el servicio no se ve afectado): marcarla sería un falso positivo.

**Datos:** los resúmenes `comparacion_dev_*.json` y `comparacion_test_*.json` que guarda `06_experimento_2_resultados` en `resultados/experimento_2/dev/` y `resultados/experimento_2/test/` (ejecuta antes el 06 con cada conjunto). Solo llevan veredictos, probabilidades, la referencia y el grupo: ni textos del modelo ni nombres de señales. **No necesita GPU.**

**Referencia: solo mis etiquetas.** La sección 2.2 comprueba ventana a ventana que la referencia de los resúmenes es exactamente la de las entradas del experimento (`ventanas_desarrollo.csv` y `ventanas_prueba.csv`, creadas por el `05_`) y se detiene si alguna no coincide.

### Los dos conjuntos de ventanas

| | **Desarrollo** | **Prueba** |
|---|---|---|
| Partición | Validación | Prueba |
| Cuántas | 29 | 40 |
| Cómo se eligieron | **A mano, por ti**, ordenando las ventanas por los valores de cada contador. Sin información de TranAD | **Sorteo** con semilla 2026, con las reglas fijadas antes de ver ninguna ventana: 20 alertas de consenso de TranAD y TranAD+ (una por sesión), 10 sin alerta de alta actividad (≥ percentil 90 de actividad) y 10 sin alerta al azar |
| Tus etiquetas | Columna `referencia` de `ventanas_desarrollo.csv` | Columna `referencia` de `ventanas_prueba.csv`, clasificadas a ciegas, sin saber el grupo |
| Uso | Ajustar los prompts y elegir variantes | Evaluar variantes ya cerradas |

### Tipos de ventana según tus etiquetas

| Tipo | Etiqueta | Acierta el modelo que responde | Desarrollo | Prueba |
|---|---|---|---|---|
| Anomalía | `anomalia` | «anomalía» | 15 | 25 |
| Anomalía mitigada | `anomalia_mitigada` | «normal» (los errores se recuperan) | 4 | 2 |
| Funcionamiento normal | `normal` | «normal» | 7 | 10 |
| Dudosa | `dudosa` | fuera del cálculo | 3 | 3 |

Una ventana sin veredicto (JSON no válido o respuesta cortada) cuenta como fallo.

**Cómo leerlo:**
- La comparación principal fijada antes de la prueba (aciertos de Qwen3.5-9B frente a Qwen2.5-7B, N1 con señales en filas, 40 ventanas de prueba) está en el `06_` y no cambia.
- Lo de este notebook es **exploratorio**: el desglose por tipo de ventana se analiza después de ver la prueba, y el conjunto de desarrollo es el que se usó para ajustar los prompts.
- Solo hay **6 anomalías mitigadas** (4 en desarrollo y 2 en prueba). Acertar en ellas solo tiene valor si el modelo sigue acertando en las anomalías: mira siempre las dos cosas juntas.
- La sección 3.5 hace muchas comparaciones a la vez: un p < 0,05 aislado puede salir por azar.

| Sección | Qué hace |
|---|---|
| 1–2 | Entorno, parámetros, carga y comprobación de tus etiquetas |
| 3 | Tablas: por variante, por tipo de ventana, la variante principal, ventana a ventana, todos los pares y cambio al girar la tabla |
| 4 | Gráficas: aciertos en anomalías, en anomalías mitigadas y en funcionamiento normal; y ventana a ventana |
| 5 | Guardado de tablas y gráficas en `resultados/experimento_2/estadisticas_<fecha>/` |

## 1. Entorno

En Colab, monta Google Drive y clona el repositorio para tener acceso a `src/`. En local, busca la carpeta `src/` subiendo desde el directorio actual.

In [ ]:
# --- Entorno: localiza src/ y, en Colab, monta Google Drive ---
import os
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/Williams-MP/TFM_VIU_2026.git"
RUTA_REPO_COLAB = "/content/TFM_VIU_2026"

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(RUTA_REPO_COLAB):
        # Mientras el repositorio sea privado hace falta un token de GitHub.
        # Si es público, basta con dejar el campo vacío.
        from getpass import getpass
        token = getpass("Token de GitHub (vacío si el repositorio es público): ")
        url = URL_REPOSITORIO.replace("https://", f"https://{token}@") if token else URL_REPOSITORIO
        subprocess.run(["git", "clone", "-q", url, RUTA_REPO_COLAB], check=True)
    ruta_src = f"{RUTA_REPO_COLAB}/src"
else:
    # Ejecución local: se sube por las carpetas hasta encontrar src/.
    directorio = os.getcwd()
    while not os.path.isdir(os.path.join(directorio, "src")):
        padre = os.path.dirname(directorio)
        if padre == directorio:
            raise FileNotFoundError("No se encuentra la carpeta src/ del repositorio.")
        directorio = padre
    ruta_src = os.path.join(directorio, "src")

if ruta_src not in sys.path:
    sys.path.append(ruta_src)

from datetime import datetime

import pandas as pd
import plotly.io as pio

from config import RUTA_EXPERIMENTO_2, RUTA_EXPERIMENTO_2_PRIVADO
from llm_lab import exp2_estadisticas as est

RUTA_ENTRADAS = f"{RUTA_EXPERIMENTO_2_PRIVADO}/entradas"   # ventanas con tu referencia (sin notas)
pio.renderers.default = "vscode"   # "colab" si lo ejecutas en Colab desde el navegador

In [ ]:
# --- Parámetros (EDITABLE) ---
COMPARACIONES = None    # None = el resumen más reciente de cada conjunto; o {"dev": "comparacion_dev_....json", "test": "..."}
MODELO_A, MODELO_B = "qwen2.5-7b", "qwen3.5-9b"       # pareja de la comparación ventana a ventana (3.4)
VARIANTE = ("N1", "senales_en_filas")                 # variante de la comparación principal
CONJUNTOS_GRAFICAS = ("dev", "test", "juntos")        # grupos de barras de 4.1–4.3 ("juntos" = desarrollo + prueba)
GUARDAR = True                                        # sección 5: guardar tablas y gráficas en Drive

# Tus etiquetas (solo lectura): la referencia de todo el notebook
VENTANAS_CSV = {"dev": f"{RUTA_ENTRADAS}/ventanas_desarrollo.csv", "test": f"{RUTA_ENTRADAS}/ventanas_prueba.csv"}

## 2. Carga y comprobación de tus etiquetas

Desarrollo: 29 ventanas (15 anomalías, 4 mitigadas, 7 normales, 3 dudosas). Prueba: 40 (25, 2, 10 y 3). Cada ventana, con 3 modelos × 4 variantes.

In [ ]:
# --- 2.1 Resúmenes del 06 (sin textos del modelo) ---
if COMPARACIONES:
    rutas = {c: f"{RUTA_EXPERIMENTO_2}/{c}/{nombre}" for c, nombre in COMPARACIONES.items()}
else:
    rutas = {c: est.ultima_comparacion(f"{RUTA_EXPERIMENTO_2}/{c}", c) for c in ["dev", "test"]}
for conjunto, ruta in rutas.items():
    print(f"{conjunto}: {os.path.basename(ruta)}")

df = est.cargar_llamadas(rutas)
display(est.resumen_conjuntos(df))

In [ ]:
# --- 2.2 La referencia es exactamente la de tus etiquetas (se detiene si alguna ventana no coincide) ---
display(est.comprobar_etiquetas(df, VENTANAS_CSV))
print("Referencia comprobada: coincide ventana a ventana con tus etiquetas")

## 3. Tablas

- **3.1** Todas las medidas por conjunto, modelo y variante.
- **3.2** Aciertos por tipo de ventana (anomalía, mitigada, normal), para cada modelo y variante, en desarrollo, en prueba y en los dos juntos.
- **3.3** La variante principal, con los tres modelos uno al lado de otro.
- **3.4** `MODELO_B` frente a `MODELO_A`, ventana a ventana: aciertos en total y por tipo de ventana (McNemar exacto).
- **3.5** Aciertos en anomalías mitigadas y en funcionamiento normal: cada par de modelos en cada variante, para ver si una ventaja se repite o es propia de una sola variante.
- **3.6** Ventanas cuyo veredicto cambia al girar la tabla (señales en filas frente a reportes en filas).
- **3.7** **Tabla para la memoria** (acompaña a las gráficas 4.1–4.3): en la variante principal, aciertos de los tres modelos por tipo de ventana y conjunto, y la comparación ventana a ventana de Qwen3.5-9B con cada uno de los otros dos. Cada celda de aciertos lleva entre corchetes su **intervalo de confianza al 95 %** (exacto de Clopper-Pearson): el rango de proporciones de acierto compatibles con lo observado; cuanto menos ventanas, más ancho. «a / b (p)» = ventanas que acierta solo Qwen3.5-9B / solo el otro modelo, y p de McNemar exacto. Recuerda que es exploratoria: el p de «Desarrollo + prueba» incluye las ventanas con las que se ajustaron los prompts.

In [ ]:
# --- 3.1 Todas las medidas por conjunto, modelo y variante ---
tabla = est.tabla_por_variante(df)
with pd.option_context("display.max_rows", 50, "display.max_columns", 30):
    display(tabla)

In [ ]:
# --- 3.2 Aciertos por tipo de ventana, según tus etiquetas ---
tabla_tipo = est.tabla_por_tipo(df)
with pd.option_context("display.max_columns", 20):
    display(tabla_tipo)

In [ ]:
# --- 3.3 Variante principal: medidas en filas y modelos en columnas ---
tabla_principal = est.tabla_modelos(tabla, *VARIANTE)
print(est.nombre_variante(*VARIANTE))
display(tabla_principal)

In [ ]:
# --- 3.4 MODELO_B frente a MODELO_A, ventana a ventana (McNemar exacto) ---
tabla_pares = est.pares(df, MODELO_A, MODELO_B, *VARIANTE)
print(f"{est.NOMBRE_MODELO[MODELO_B]} frente a {est.NOMBRE_MODELO[MODELO_A]} · {est.nombre_variante(*VARIANTE)}")
display(tabla_pares)

In [ ]:
# --- 3.5 Aciertos en mitigadas y en normales: cada par de modelos en cada variante ---
tabla_mitigadas_pares = est.pares_todas_las_variantes(df, "Aciertos en anomalías mitigadas")
tabla_normales_pares = est.pares_todas_las_variantes(df, "Aciertos en funcionamiento normal")
with pd.option_context("display.max_rows", 100):
    print("Aciertos en anomalías mitigadas")
    display(tabla_mitigadas_pares)
    print("Aciertos en funcionamiento normal")
    display(tabla_normales_pares)
print(f"{len(tabla_mitigadas_pares) + len(tabla_normales_pares)} comparaciones: un p < 0,05 aislado puede deberse al azar")

In [ ]:
# --- 3.6 Veredictos que cambian al girar la tabla (todas las ventanas, también las dudosas) ---
tabla_orientacion = est.cambios_orientacion(df)
display(tabla_orientacion)

In [ ]:
# --- 3.7 Tabla para la memoria: aciertos por tipo de ventana y comparación de MODELO_B con los otros dos ---
tabla_memoria = est.tabla_resumen(df, *VARIANTE, modelo=MODELO_B)
print(f"{est.nombre_variante(*VARIANTE)} · aciertos de ventanas de ese tipo según tus etiquetas")
display(tabla_memoria)

## 4. Gráficas

Las tres primeras tienen la misma forma: un panel por variante, un grupo de barras por conjunto y una barra por modelo. La altura es el % de aciertos y encima va «aciertos/ventanas»; si pone 0/n, el modelo no acertó ninguna. Al pie de cada gráfica se explica de dónde sale cada conjunto y cómo se eligió.

- **4.1** Aciertos en las ventanas que etiquetaste como **anomalía** (acierta si responde «anomalía»).
- **4.2** Aciertos en las **anomalías mitigadas** (acierta si responde «normal»).
- **4.3** Aciertos en el **funcionamiento normal** (acierta si responde «normal»).
- **4.4** Ventana a ventana, en la variante principal: qué mitigadas y qué normales acierta cada modelo, con tu etiqueta junto a cada ventana.
- **4.5** Las gráficas 4.1–4.3 en formato **2×2** (dos filas de dos variantes), más compactas para la memoria, y una cuarta con los **aciertos en total** (todas las ventanas menos las dudosas).

Para pasar una gráfica a la memoria: botón de la cámara (arriba a la derecha de la gráfica) → descarga un PNG a 3 veces la resolución de pantalla.

In [ ]:
# --- 4.1 Aciertos en anomalías ---
fig_anomalias = est.grafica_aciertos_tipo(df, "anomalia", CONJUNTOS_GRAFICAS)
fig_anomalias.show(config=est.CONFIG_PNG)

In [ ]:
# --- 4.2 Aciertos en anomalías mitigadas ---
fig_mitigadas = est.grafica_aciertos_tipo(df, "mitigada", CONJUNTOS_GRAFICAS)
fig_mitigadas.show(config=est.CONFIG_PNG)

In [ ]:
# --- 4.3 Aciertos en funcionamiento normal ---
fig_normales = est.grafica_aciertos_tipo(df, "normal", CONJUNTOS_GRAFICAS)
fig_normales.show(config=est.CONFIG_PNG)

In [ ]:
# --- 4.4 Ventana a ventana: mitigadas y normales en la variante principal ---
fig_por_ventana = est.grafica_sin_anomalia_por_ventana(df, *VARIANTE)
fig_por_ventana.show(config=est.CONFIG_PNG)

In [ ]:
# --- 4.5 Las gráficas 4.1–4.3 en formato 2x2 (para la memoria) y la de aciertos en total ---
fig_total_2x2 = est.grafica_aciertos_tipo(df, "total", CONJUNTOS_GRAFICAS, disposicion="2x2")
fig_anomalias_2x2 = est.grafica_aciertos_tipo(df, "anomalia", CONJUNTOS_GRAFICAS, disposicion="2x2")
fig_mitigadas_2x2 = est.grafica_aciertos_tipo(df, "mitigada", CONJUNTOS_GRAFICAS, disposicion="2x2")
fig_normales_2x2 = est.grafica_aciertos_tipo(df, "normal", CONJUNTOS_GRAFICAS, disposicion="2x2")
for fig in (fig_total_2x2, fig_anomalias_2x2, fig_mitigadas_2x2, fig_normales_2x2):
    fig.show(config=est.CONFIG_PNG)

## 5. Guardado

Las tablas en CSV y las gráficas en HTML (se abren sin conexión) en `resultados/experimento_2/estadisticas_<fecha>/`. Solo llevan cifras e identificadores de ventana: se pueden compartir.

In [ ]:
# --- 5. Tablas y gráficas en resultados/experimento_2 ---
if GUARDAR:
    carpeta = f"{RUTA_EXPERIMENTO_2}/estadisticas_{datetime.now():%Y%m%d_%H%M}"
    os.makedirs(carpeta, exist_ok=True)
    tablas = {"por_variante": tabla, "por_tipo": tabla_tipo, "variante_principal": tabla_principal.reset_index(),
              "pares": tabla_pares, "mitigadas_todos_los_pares": tabla_mitigadas_pares,
              "normales_todos_los_pares": tabla_normales_pares, "cambios_orientacion": tabla_orientacion,
              "tabla_memoria": tabla_memoria}
    for nombre, t in tablas.items():
        t.to_csv(f"{carpeta}/{nombre}.csv", index=(nombre in ("por_tipo", "cambios_orientacion", "tabla_memoria")))
    figuras = {"aciertos_anomalias": fig_anomalias, "aciertos_mitigadas": fig_mitigadas,
               "aciertos_normales": fig_normales, "mitigadas_y_normales_por_ventana": fig_por_ventana,
               "aciertos_total_2x2": fig_total_2x2, "aciertos_anomalias_2x2": fig_anomalias_2x2, "aciertos_mitigadas_2x2": fig_mitigadas_2x2,
               "aciertos_normales_2x2": fig_normales_2x2}
    for nombre, fig in figuras.items():
        # Autónomo (funciona sin conexión); el botón de la cámara descarga un PNG a 3x
        fig.write_html(f"{carpeta}/{nombre}.html", include_plotlyjs=True, config=est.CONFIG_PNG)
    print("Guardado en:", carpeta)
    print("Ficheros:", sorted(os.listdir(carpeta)))
else:
    print("No se guarda nada (GUARDAR = False)")